#    Libraries

In [45]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

#    Data Preparation

In [46]:
transform = transforms.Compose([
    transforms.Resize((128, 128)), # تصغير الصور عشان التدريب يكون أسرع
    transforms.ToTensor(),         # تحويل الصورة لمصفوفة أرقام
])

In [47]:
dataset = datasets.ImageFolder(root=r'F:\Documents\ML & AI\Projects\Game Scene Classifier\dataset', transform=transform)

In [48]:
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

In [49]:
print(f"Total images found: {len(dataset)}")
print(f"Classes: {dataset.classes}")

Total images found: 971
Classes: ['Elden_Ring', 'RDR2', 'RE']


#    CNN Architecture

In [50]:
class GameClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        #(Feature Extraction)
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2), # الحجم هيبقى 64x64
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)  # الحجم هيبقى 32x32
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 32, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


#    Hardware Training Settings

In [51]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

Training on: cpu


In [52]:
# 3 Classes for 3 Games
model = GameClassifier(num_classes=3).to(device)

In [53]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

#    Model Training

In [54]:
epochs = 5

for epoch in range(epochs):
    total_loss = 0
    for images, labels in dataloader:
        # نقل الصور لكارت الشاشة
        images, labels = images.to(device), labels.to(device)
        
        # 1. Prediction
        predictions = model(images)
        
        # 2. Loss
        loss = criterion(predictions, labels)
        
        # 3. Backward pass
        optimizer.zero_grad() # Zero the weigts
        loss.backward()       # Derivatives calculation
        optimizer.step()      # Updating The Weights
        
        total_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(dataloader):.4f}")

Epoch [1/5], Loss: 0.6790
Epoch [2/5], Loss: 0.1906
Epoch [3/5], Loss: 0.1051
Epoch [4/5], Loss: 0.0465
Epoch [5/5], Loss: 0.0506


In [55]:
# Saving The weights 
torch.save(model.state_dict(), 'game_classifier.pth')
print("تم حفظ الموديل بنجاح!")

تم حفظ الموديل بنجاح!


#    Testing

In [57]:
from PIL import Image

# 1. قراءة الصورة وتجهيزها (بنستخدم نفس الـ transform اللي عرفناه فوق)
image_path = r'F:\Documents\ML & AI\Projects\Game Scene Classifier\rdr2_test.jpg' # اتأكد إن الصورة في نفس الفولدر
image = Image.open(image_path).convert('RGB')

# إضافة بُعد وهمي ونقل الصورة لكارت الشاشة
input_tensor = transform(image).unsqueeze(0).to(device)

# 2. تحويل الموديل لوضع الاختبار
model.eval()

# 3. استخراج التوقع
with torch.no_grad():
    output = model(input_tensor)
    _, predicted_idx = torch.max(output, 1)

# هنستخدم dataset.classes اللي في الذاكرة أوتوماتيك
print(f"🔥 The Game is: {dataset.classes[predicted_idx.item()]}")

🔥 The Game is: RDR2
